# BirdCLEF+ 2026 — Google Perch Baseline

Google Perch (`bird-vocalization-classifier v4`) zero-shot baseline.

**Strategy**:
- Load Perch TF SavedModel from Kaggle dataset
- Map 9736 Perch eBird labels → 234 competition species (birds only)
- Sliding window inference (5s windows) on test soundscapes
- Non-Aves species (insects, amphibians, etc.) → uniform score

**Inputs needed**:
- `/kaggle/input/birdclef-2026/` — competition data
- `/kaggle/input/birdclef2026-perch/model/` — Perch TF SavedModel
- `/kaggle/input/birdclef2026-perch/label.csv` — Perch label list

In [ ]:
import os
import glob
import math
import subprocess
import warnings
import numpy as np
import pandas as pd
import soundfile as sf
from pathlib import Path
from tqdm.auto import tqdm

import tensorflow as tf
warnings.filterwarnings('ignore')

# ── CPU threading: use all available cores ─────────────────────────────────────
_ncpu = os.cpu_count() or 4
tf.config.threading.set_intra_op_parallelism_threads(_ncpu)
tf.config.threading.set_inter_op_parallelism_threads(_ncpu)
tf.config.set_visible_devices([], 'GPU')   # CPU-only notebook
print(f'TF {tf.__version__}  |  threads={_ncpu}')

# ── Debug: show ALL mounted datasets ──────────────────────────────────────────
print('=== /kaggle/input/ top-level ===')
for name in sorted(os.listdir('/kaggle/input')):
    p = Path('/kaggle/input') / name
    try:
        children = list(p.iterdir())
        print(f'  {name}/  ({len(children)} items)')
        for c in sorted(children)[:10]:
            print(f'    {c.name}/')
    except Exception as e:
        print(f'  {name}/  ERROR: {e}')

# ── Paths ──────────────────────────────────────────────────────────────────────
OUTPUT      = Path('/kaggle/working')

SAMPLE_RATE = 32_000
WIN_SAMPLES = 5 * SAMPLE_RATE
WIN_STRIDE  = 5 * SAMPLE_RATE
BATCH_SIZE  = 16   # larger batches → better CPU utilisation

# Auto-detect Perch model path (search all of /kaggle/input recursively)
_pb_hits = subprocess.run(['find', '/kaggle/input', '-name', 'saved_model.pb'], capture_output=True, text=True).stdout.strip().split('\n')
_pb_hits = [x for x in _pb_hits if x]
print(f'\nsaved_model.pb hits: {_pb_hits}')
assert _pb_hits, 'saved_model.pb not found anywhere under /kaggle/input/'
PERCH_MODEL = Path(_pb_hits[0]).parent
print(f'Perch model path: {PERCH_MODEL}')

# Auto-detect label.csv
_label_hits = subprocess.run(['find', '/kaggle/input', '-name', 'label.csv'], capture_output=True, text=True).stdout.strip().split('\n')
_label_hits = [x for x in _label_hits if x]
print(f'label.csv hits: {_label_hits}')
assert _label_hits, 'label.csv not found'
PERCH_LABELS = Path(_label_hits[0])
print(f'Perch labels: {PERCH_LABELS}')


In [ ]:
# ── Load model ─────────────────────────────────────────────────────────────────
print('Loading Perch SavedModel …')
model = tf.saved_model.load(str(PERCH_MODEL))
print('  OK')

# Trace + warmup with tf.function so subsequent batches use a compiled graph
@tf.function(input_signature=[tf.TensorSpec(shape=[None, WIN_SAMPLES], dtype=tf.float32)])
def run_model(batch):
    return model.infer_tf(batch)

print('Warming up (tracing tf.function) …')
dummy = tf.zeros([1, WIN_SAMPLES])
logits_test, _ = run_model(dummy)
N_PERCH = logits_test.shape[1]
print(f'  Perch output classes: {N_PERCH}')


In [ ]:
# ── Build species → Perch index mapping ────────────────────────────────────────

# Auto-detect competition data path (search up to 2 levels deep)
COMP_DATA = None
for root in [Path('/kaggle/input'), Path('/kaggle/input/competitions'), Path('/kaggle/input/datasets')]:
    if not root.exists():
        continue
    for p in sorted(root.iterdir()):
        if (p / 'sample_submission.csv').exists():
            COMP_DATA = p
            print(f'Found competition data at: {COMP_DATA}')
            break
    if COMP_DATA:
        break

if COMP_DATA is None:
    import subprocess
    tree = subprocess.run(['find', '/kaggle/input', '-name', 'sample_submission.csv'], capture_output=True, text=True)
    print('find result:', tree.stdout or '(nothing)')
    raise FileNotFoundError('Cannot find sample_submission.csv anywhere under /kaggle/input')

# Competition species (234 labels = column names after row_id)
sub_df     = pd.read_csv(COMP_DATA / 'sample_submission.csv', nrows=0)
COMP_SPECIES = list(sub_df.columns[1:])

# Taxonomy to know which species are Aves (birds) vs invertebrates
tax_df = pd.read_csv(COMP_DATA / 'taxonomy.csv')
is_aves = dict(zip(tax_df['primary_label'], tax_df['class_name'] == 'Aves'))

# Perch eBird label list
perch_labels = pd.read_csv(PERCH_LABELS, skiprows=1, header=None, names=['ebird_code'])
perch_idx    = {code: i for i, code in enumerate(perch_labels['ebird_code'])}

# Map each competition species to its Perch index (-1 = not in Perch)
comp_to_perch = []
n_mapped = 0
for sp in COMP_SPECIES:
    if is_aves.get(sp, False) and sp in perch_idx:
        comp_to_perch.append(perch_idx[sp])
        n_mapped += 1
    else:
        comp_to_perch.append(-1)   # non-Aves or unknown bird → use uniform score

comp_to_perch = np.array(comp_to_perch, dtype=np.int32)

print(f'Competition species: {len(COMP_SPECIES)}')
print(f'Mapped to Perch:     {n_mapped} (of {(np.array([is_aves.get(s,False) for s in COMP_SPECIES])).sum()} Aves)')
print(f'Unmapped (→ 0):      {len(COMP_SPECIES) - n_mapped}')


In [ ]:
# ── Inference helpers ──────────────────────────────────────────────────────────

# Precompute vectorised mapping indices once (avoids Python for-loop in hot path)
_valid_mask   = comp_to_perch >= 0                                    # (234,) bool
_safe_perch   = np.where(_valid_mask, comp_to_perch, 0).astype(np.int32)   # safe idx


def load_audio(path: Path) -> np.ndarray:
    """Load OGG/WAV at native sample rate, return float32 mono normalised [-1,1]."""
    data, sr = sf.read(str(path), dtype='float32', always_2d=True)
    if data.shape[1] > 1:
        data = data.mean(axis=1)
    else:
        data = data[:, 0]
    if sr != SAMPLE_RATE:
        raise ValueError(f'Expected 32kHz, got {sr}Hz for {path}')
    mx = np.abs(data).max()
    if mx > 0:
        data = data / mx
    return data


def infer_windows(waveform: np.ndarray) -> np.ndarray:
    """Run Perch inference on all 5s windows of a soundscape.

    Returns:
        probs: (n_windows, 234) float32 — sigmoid probabilities per competition species
    """
    total = len(waveform)
    n_win = math.ceil(total / WIN_SAMPLES)

    pad = n_win * WIN_SAMPLES - total
    if pad > 0:
        waveform = np.concatenate([waveform, np.zeros(pad, dtype=np.float32)])

    windows = waveform.reshape(n_win, WIN_SAMPLES).astype(np.float32)

    all_perch_probs = []
    for i in range(0, n_win, BATCH_SIZE):
        batch = tf.constant(windows[i: i + BATCH_SIZE])
        logits, _ = run_model(batch)                     # uses compiled tf.function
        probs = tf.sigmoid(logits).numpy()
        all_perch_probs.append(probs)

    perch_probs = np.concatenate(all_perch_probs, axis=0)    # (n_win, N_PERCH)

    # Vectorised mapping: single numpy fancy-index instead of 234-iteration for-loop
    comp_probs = perch_probs[:, _safe_perch]                  # (n_win, 234)
    comp_probs[:, ~_valid_mask] = 0.0                         # zero unmapped species

    return comp_probs   # (n_windows, 234)


def row_ids_for(stem: str, n_win: int) -> list:
    """Generate row_ids matching competition format: <stem>_<end_seconds>."""
    return [f'{stem}_{(i + 1) * 5}' for i in range(n_win)]


In [ ]:
# ── Run inference on all test soundscapes ─────────────────────────────────────
test_files = sorted((COMP_DATA / 'test_soundscapes').glob('*.ogg'))
print(f'Test soundscapes: {len(test_files)}')

rows = []
for fpath in tqdm(test_files, desc='Inference'):
    try:
        waveform = load_audio(fpath)
        probs    = infer_windows(waveform)          # (n_win, 234)
        n_win    = len(probs)
        ids      = row_ids_for(fpath.stem, n_win)
        for rid, p in zip(ids, probs):
            rows.append([rid] + p.tolist())
    except Exception as e:
        print(f'  [warn] {fpath.name}: {e}')
        # Fall back to uniform score row
        fallback = 1.0 / len(COMP_SPECIES)
        for t in range(5, 65, 5):
            rows.append([f'{fpath.stem}_{t}'] + [fallback] * len(COMP_SPECIES))

print(f'Total rows: {len(rows)}')

In [ ]:
# ── Build & validate submission ────────────────────────────────────────────────
sub = pd.DataFrame(rows, columns=['row_id'] + COMP_SPECIES)

# Ensure all expected row_ids from sample_submission are present
sample_sub = pd.read_csv(COMP_DATA / 'sample_submission.csv')
sub = sample_sub[['row_id']].merge(sub, on='row_id', how='left')

# Fill any missing rows with uniform scores
uniform = 1.0 / len(COMP_SPECIES)
sub[COMP_SPECIES] = sub[COMP_SPECIES].fillna(uniform)

print(f'Submission shape: {sub.shape}')
print(f'Missing values:   {sub.isnull().sum().sum()}')
print(sub.head(3))

out_path = OUTPUT / 'submission.csv'
sub.to_csv(out_path, index=False)
print(f'\nSaved → {out_path}')